# Convert CSV to NDJSON with chDB

chDB is ClickHouse as an in-process Python module. The SQL is identical to the `clickhouse local` CLI version: `SELECT` from the CSV, `FORMAT JSONEachRow`, write the bytes to a file.

Run `./generate.sh` first to create `./data/events.csv`.

In [ ]:
import os
import chdb

os.chdir("data")

In [ ]:
# Convert: read the CSV, render every row as a JSON object, write NDJSON.
ndjson = chdb.query("SELECT * FROM file('events.csv') FORMAT JSONEachRow").bytes()

with open("events_chdb.ndjson", "wb") as f:
    f.write(ndjson)

print("wrote events_chdb.ndjson")

In [ ]:
# Look at the first few lines.
with open("events_chdb.ndjson") as f:
    for _ in range(5):
        print(f.readline().rstrip())

In [ ]:
# Read the NDJSON straight back to prove the round-trip.
print(
    chdb.query(
        "SELECT country, count() AS events, round(sum(value),2) AS total "
        "FROM file('events_chdb.ndjson') GROUP BY country ORDER BY total DESC"
    )
)